# 02_check_jalc_extract

MongoDB の JaLC 文書から保存フィールドと token fields を作れるか確認する。

In [ ]:
# Notebook から src 配下の自作モジュールを import できるようにする設定
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
# MongoDB と抽出関数のインポート
from pprint import pprint

from pymongo import MongoClient

from tanigawa_shoshi.config import MONGODB_COLLECTION, MONGODB_DATABASE, MONGODB_URL
from tanigawa_shoshi.jalc_extract import (
    build_solr_document,
    extract_authors,
    extract_doi,
    extract_first_author,
    extract_journals,
    extract_page,
    extract_titles,
    extract_volume,
    extract_year,
    has_required_fields,
)

In [ ]:
# MongoDB に接続して、JaLC 文書を少件数取得する
# creator_listがないものがあるらしくそれを取得段階で弾く
client = MongoClient(MONGODB_URL)
collection = client[MONGODB_DATABASE][MONGODB_COLLECTION]

sample_docs = list(
    collection.find(
        {"content_type": "JA", "creator_list": {"$ne": None}},
        {
            "doi": 1,
            "creator_list": 1,
            "title_list": 1,
            "journal_title_name_list": 1,
            "publication_date": 1,
            "volume": 1,
            "issue": 1,
            "first_page": 1,
            "last_page": 1,
        },
    ).limit(3)
)

print(f"sample docs: {len(sample_docs)}")

In [ ]:
# 元文書の必要箇所と抽出結果を確認する
for index, doc in enumerate(sample_docs, start=1):
    print("=" * 80)
    print(f"sample {index}")
    print(f"doi: {doc.get('doi')}")
    print(f"has_required_fields: {has_required_fields(doc)}")
    print("\n[raw keys]")
    pprint({
        "creator_list": doc.get("creator_list"),
        "title_list": doc.get("title_list"),
        "journal_title_name_list": doc.get("journal_title_name_list"),
        "publication_date": doc.get("publication_date"),
        "volume": doc.get("volume"),
        "issue": doc.get("issue"),
        "first_page": doc.get("first_page"),
        "last_page": doc.get("last_page"),
    })

    print("\n[extracted fields]")
    pprint({
        "doi": extract_doi(doc),
        "authors": extract_authors(doc),
        "first_author": extract_first_author(doc),
        "title": extract_titles(doc),
        "journal": extract_journals(doc),
        "year": extract_year(doc),
        "volume": extract_volume(doc),
        "page": extract_page(doc),
    })

    print("\n[solr document]")
    pprint(build_solr_document(doc))
